In [1]:
import os
import subprocess
import sys
import matplotlib.pyplot as plt
import rioxarray as rxr
import matplotlib
import numpy as np
import imageio
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor
import utm


In [2]:
(utm_east, utm_north, zone, _) = utm.from_latlon(43.639722233573, -112.5833333442)
print (utm_east, utm_north)

#-112.88305556667 -112.5833333442
#43.340000011111 43.639722233573

372291.23668458115 4833076.781052108


### Enter date

First, please enter date-related parameters

In [3]:
year = 2007

month = 2

day = 24

hour = 12

minute = "00"

#Set duration of simulation in seconds
sim_duration = 600

#If restart file exists, use it (NO/YES)
restart_run = "NO"

#If using the restart file, continue from existing time, or from 0 (NO/YES)
reset_time = "NO"

### Specify grid and topography parameters

Please specify grid spacing and origin in UTM, as well as if you are grabbing the topography from a file, or if you are creating a simple map with a slope,

In [4]:
num_x = 600

num_y = 600

num_z = 8

#Array of layers to calculate (m)
z_layers = [0, 1, 4, 8, 10, 100, 200, 300]

#grid spacing along x direction (m)
dx = 30

#grid spacing along y direction (m)
dy = 30

x_utm = 347362
y_utm = 4800294

utm_zone = "12"

#Get the topography from a file
extract_topo_from_file =  "YES"

#Set height of bottom left corner of topography, if not using file
z_origin = 2000.0

#Slope of the flat topography in the x direction (degrees)
x_slope = 0.0

#Slope of the flat topography in the y direction (degrees)
y_slope = 0.0

### Enter model calculation parameters

Please choose options related to the treatment of the model as a gas (with no settling velocity) or as particles (settling velocity calculated in accordance with specified parameters.

In [5]:
#Options are GAS, PARTICLES
dispersion_type = "GAS"

#If PARTICLES is chosen, set the next 4, otherwise ignore

#Set particle diameter (m)
particle_diameter = 1e-5

#Set particle density (kg/m^3)
particle_density = 2500

#Set if particle shape parameter is used in calculating setting velocity (use=1, ignored=0)
particle_shape_parameter = 1

#Set model for particle velocity calculation. Choices are ("ARASTOOPOUR"/"GANSER"/"WILSON_HUANG")
particle_model_vset = "GANSER"

### Enter meteorology terms

Please set the wind model and related coefficients/constants.

In [6]:
#Set wind model (SIMILARITY/DIAGNO), from file/diagno utility
wind_model = "SIMILARITY"

#Set horizontal turbulence model, choose from: CONSTANT, SMAGORINSKY
horizontal_turb_model = "SMAGORINSKY"

#Set vertical turbulence model (CONSTANT/SIMILARITY)
vertical_turb_model = "SIMILARITY"

#Set roughness model (UNIFORM/MATRIX). MATRIX requires a file to specify roughness length
roughness_model = "UNIFORM"

#Set roughness length if uniform
roughness_level = .18

#Set horizontal diffusion coefficient if the model is set to constant
diff_coeff_horizontal = 10

#Set vertical diffusion coefficient if the model is set to constant
diff_coeff_vertical = 10

### Enter file paths and output specifications

Please, if applicable, set paths for relevant model files as well as verboseness and inclusion of additional data in output files.

In [7]:
#Only needed if extract_topo_from_file is set to "YES"
topography_file = "ESRP.grd"

#Only neeeded when roughness_model is set to "MATRIX"
roughness_file = "roughness.grd"

#Only needed if run is a restart
restart_file = "restart.dat"

source_file = "ESRP_source.dat"

#Only needed when wind model is set to "SIMILARITY"
wind_file = "ESRP_winds.dat"

#Only needed when wind model set to "DIAGNO"
diagno_file = "diagno.out"

#Directory that output files will be output to
output_directory = "outfiles"

#Set 0 for low verbosity, 1 for high
log_verbosity = 0

#Set output interval in seconds
interval = 30
 
#Output a file at each interval for the wind in the x direction in m/s ("YES"/"NO") 
output_u_velocity = "YES"

#Output a file at each interval for the wind in the y direction in m/s ("YES"/"NO") 
output_v_velocity = "YES"

#Output a file at each interval for the concentration of gas direction in kg/m^3 ("YES"/"NO") 
output_conc = "YES"

In [8]:
inp_file = f"""DISGAS INPUT FILE

TIME
  YEAR   = {year}
  MONTH  = {month}
  DAY    = {day}
  HOUR   = {hour}
  MINUTE = {minute}
  SIMULATION_INTERVAL_(SEC) = {sim_duration}
  RESTART_RUN = {restart_run}
  RESET_TIME  = {reset_time}

GRID
  NX     = {num_x}
  NY     = {num_y}
  NZ     = {num_z}
  Z_LAYERS_(M) = {str(z_layers)[1:-1].replace(",",".")+"."}
  DX_(M) = {dx}.
  DY_(M) = {dy}.
  X_ORIGIN_(UTM_M) =  {x_utm}
  Y_ORIGIN_(UTM_M) = {y_utm}
  UTM_ZONE = {utm_zone}

PROPERTIES
  DISPERSION_TYPE = {dispersion_type}            (GAS/PARTICLES)
  PARTICLE_DIAMETER = {particle_diameter}         (DIAMETER IN M)
  PARTICLE_DENSITY =  {particle_density}.         (KG/M3)
  PARTICLE_SHAPE_PARAMETER = {particle_shape_parameter}.     ([0-1])
  PARTICLE_MODEL_VSET = {particle_model_vset}      (ARASTOOPOUR/GANSER/WILSON_HUANG)

TOPOGRAPHY
  EXTRACT_TOPOGRAPHY_FROM_FILE = {extract_topo_from_file}    (YES/NO)
  Z_ORIGIN_(M) = {z_origin}
  X_SLOPE_(DEG) = {x_slope}
  Y_SLOPE_(DEG) = {y_slope}

METEO
  WIND_MODEL            = {wind_model}    (SIMILARITY/DIAGNO)
  HORIZONTAL_TURB_MODEL = {horizontal_turb_model}   (CONSTANT/SMAGORINSKY)
  VERTICAL_TURB_MODEL   = {vertical_turb_model}    (CONSTANT/SIMILARITY)
  ROUGHNESS_MODEL       = {roughness_model}       (UNIFORM/MATRIX)
  ROUGHNESS_LENGTH      = {roughness_level}          (Roughness length if UNIFORM)
  DIFF_COEFF_HORIZONTAL = {diff_coeff_horizontal}           (Only for turbulence model=0)
  DIFF_COEFF_VERTICAL   = {diff_coeff_vertical}           (Only for turbulence model=0)

 FILES                                        (Relative or absolute file paths)
   TOPOGRAPHY_FILE_PATH   = {topography_file}  (Only used if topography is extracted from a file)  
   ROUGHNESS_FILE_PATH    = {roughness_file}   (Only used if roughness model is MATRIX)
   RESTART_FILE_PATH      = {restart_file}     (Only used if the run is a restart)
   SOURCE_FILE_PATH       = {source_file}
   WIND_FILE_PATH         = {wind_file}       (Only used with SIMILARITY wind model)
   DIAGNO_FILE_PATH       = {diagno_file}      (Only used with DIAGNO wind model)
   OUTPUT_DIRECTORY       = {output_directory}

OUTPUT
  LOG_VERBOSITY_LEVEL  = {log_verbosity}                 (0/1)
  OUTPUT_GRD_TYPE      = ASCII             (ASCII/BINARY)
  OUTPUT_INTERVAL_(SEC) = {interval}               Number of seconds between prints
  OUTPUT_U_VELOCITY = {output_u_velocity}                  (YES/NO)
  OUTPUT_V_VELOCITY = {output_v_velocity}                  (YES/NO)
  OUTPUT_W_VELOCITY = NO                  (YES/NO)
  OUTPUT_CONCENTRATION = {output_conc}               (YES/NO)
  OUTPUT_LAYERS = ALL                      (ALL/List_of_layers)
  """
with open("disgas.inp",'w') as in_file:
    in_file.write(inp_file)

In [9]:
## This cell will run the Disgas executable program. It assumes you aleady have it in the /home/jovyan/Disgas folder. If not, first do "victor setup" to install the Disgas program

subprocess.run("/home/jovyan/Disgas/disgas disgas.inp", shell=True)


----------------
 DISGAS PROGRAM 
----------------
                
  Starting date 12-08-2025 at time 23:47:41

------------
 INPUT DATA 
------------

TIME
  Initial time    :   24 FEB 2007 at 12 h 0000 s
  Final   time    :   24 FEB 2007 at 12 h 0600 s
  Time increment  :    0.2 h (       600.0 s)
  Restart         :   No

GRID
  Bottom-left corner : ( 347362.0  4800294.0)
  Top-right   corner : ( 365332.0  4818264.0)
  Number points x    :       600
  Number points y    :       600
  Number points z    :         8
  X  grid spacing    :      30.0
  Y  grid spacing    :      30.0

TOPOGRAPHY
  Min. topography    :    1455.0
  Max. topography    :    1946.1

OUTPUT
  Time interval (sec):       30.0
  Dump directory     : outfiles
  Output restart file: yes
  Output domain      : no
  Output source      : no
  Output u-velocity  : yes
  Output v-velocity  : yes
  Output w-velocity  : no 
  Output ground_load : no 
  Output concentr.   : yes

----------------
 TIME EVOLUTION 
--------

KeyboardInterrupt: 

### Output Visualization

Please choose the layer you would like to visualize.

In [ ]:
layer = 4

In [ ]:
num_outputs = int(np.floor(sim_duration/interval))+1
save_index = 0

create_gif = True

for i in range(num_outputs):
    plotname = "".join(('c_',str(layer).zfill(3),"_",str(i).zfill(6),'.grd'))
    dem_check = rxr.open_rasterio("".join(("./outfiles/",plotname))).drop('band')[0].rename({'x':'easting', 'y':'northing'})
    dem_check = dem_check.where(dem_check > 0)
    if dem_check.max().isnull() and dem_check.min().isnull():
        continue
    plt.clf()
    dem_check.plot(norm=matplotlib.colors.LogNorm(vmin=dem_check.min(),vmax=dem_check.max()),cmap="bone")
    plt.savefig("".join(("gas_",str(save_index))))
    save_index += 1
if create_gif:
    frames = []
    for i in range(save_index):
        image = imageio.v2.imread(f'gas_{i}.png')
        frames.append(image)
    imageio.mimsave("gasses.gif", # output gif
    frames,          # array of input frames
    fps = 10)